<div style="text-align: center; font-weight: bold;">
    <h1>EHR Preprocessing 1: Data Cleaning</h1>
    <h4>Author: Vidul Ayakulangara Panickan</h4>
</div>

**Goal:** In this notebook, we will clean five raw MIMIC-IV datasets (diagnoses, HCPCS procedures, ICD procedures, medications, and labs) into a standardized four-column format: `subject_id`, `date`, `code`, and `coding_system`. By the end, every dataset will share the same schema and be saved as patient-level batch files, ready for code rollup in the next notebook.

**Recap:** In the [Getting Started](github_repo/TUTORIAL/EHR_Preprocessing_Getting_Started.ipynb) notebook, we explored the MIMIC-IV data structure, examined patient demographics, and generated summary statistics for each data type. We identified the three key data elements every EHR record needs: a **Patient ID** (`subject_id`), a **Clinical Code** (e.g., `icd_code`), and a **Timestamp** (e.g., `admittime`). Now we will clean and standardize these elements across all five datasets.

> **Full executable notebook:** [github_repo/TUTORIAL/EHR_Preprocessing_1_Data_Cleaning.ipynb](github_repo/TUTORIAL/EHR_Preprocessing_1_Data_Cleaning.ipynb)

## 1. Setup

We begin by importing libraries and configuring our workspace. Update `base_dir` below to point to your `EHR_TUTORIAL_WORKSPACE` folder.

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Update this path to your EHR_TUTORIAL_WORKSPACE location
base_dir = "EHR_TUTORIAL_WORKSPACE"

Next, we create the output directory where all cleaned batch files will be saved.

In [ ]:
cleaned_data_dir = os.path.join(base_dir, 'processed_data', 'step3_cleaned_rawdata')
os.makedirs(cleaned_data_dir, exist_ok=True)

print(f"Output directory: {cleaned_data_dir}")

## 2. What Data Cleaning Involves

Raw EHR data is messy. Before we can analyze it, we need to apply five cleaning steps to each dataset:

1. **Assess Missingness** -- Check for missing values in critical columns. If few records are missing, we drop them. If many are missing, we investigate further.
2. **Filter Columns** -- Keep only the columns we need (patient ID, code, timestamp) and discard the rest to save memory.
3. **Standardize Schema** -- Rename columns to a common format (`subject_id`, `date`, `code`, `coding_system`) so all datasets look the same.
4. **Identify Erroneous Data** -- Flag records with invalid dates or impossible values (not applicable to MIMIC-IV; see the note at the end of this notebook).
5. **Remove Duplicates** -- Drop exact duplicate rows to prevent double-counting in downstream analysis.

We will walk through each step on the **diagnoses** dataset first, then wrap everything into a reusable function for the remaining datasets.

Here are the five MIMIC-IV datasets we will clean in this notebook. Notice that each has a different code column and timestamp column, but they will all end up with the same four-column schema.

| Dataset | Source File | Code Column | Date Column | Coding System |
|---------|-----------|-------------|-------------|---------------|
| Diagnoses | `diagnoses_icd.csv` + `admissions.csv` | `icd_code` | `admittime` (from admissions) | ICD9 / ICD10 |
| Procedures (HCPCS) | `hcpcsevents.csv` | `hcpcs_cd` | `chartdate` | HCPCS |
| Procedures (ICD) | `procedures_icd.csv` | `icd_code` | `chartdate` | ICDPROC9 / ICDPROC10 |
| Medications | `prescriptions.csv` | `ndc` | `starttime` | NDC |
| Labs | `labevents.csv` | `itemid` | `charttime` | ITEMID |

## 3. Step-by-Step Walkthrough: Cleaning Diagnoses

We will walk through every cleaning step on the diagnoses data. Once you see the logic, we will wrap it into a function and apply it to the other four datasets.

### 3a. Loading the Data

Diagnoses are unique among our five datasets: the diagnosis codes live in `diagnoses_icd.csv`, but the timestamps live in a separate file, `admissions.csv`. We need to merge these two tables to attach a date to each diagnosis record.

In [ ]:
hosp_dir = os.path.join(base_dir, "raw_data", "structured_data", "physionet.org", "files", "mimiciv", "3.1", "hosp")

diagnoses_icd = pd.read_csv(os.path.join(hosp_dir, "diagnoses_icd.csv"), dtype=str)
admissions = pd.read_csv(os.path.join(hosp_dir, "admissions.csv"), dtype=str)

print(f"Diagnoses shape: {diagnoses_icd.shape}")
print(f"Diagnoses columns: {list(diagnoses_icd.columns)}")
print(f"\nAdmissions shape: {admissions.shape}")
print(f"Admissions columns: {list(admissions.columns)}")

The diagnoses table has `subject_id`, `hadm_id` (hospital admission ID), `icd_code`, and `icd_version`. The admissions table has `admittime`, which is the timestamp we need. We will merge them on `subject_id` and `hadm_id`.

### 3b. Assessing Missingness

Before merging, we check for missing values in both tables. We do this because rows missing a patient ID, code, or date are unusable for downstream analysis. If missingness is small, we drop those rows. If it is substantial, we need to investigate further since systematic missingness may indicate data quality issues.

In [ ]:
print("Diagnoses Table - Missing Values")
print(diagnoses_icd.isna().sum())

print("\nAdmissions Table - Missing Values")
print(admissions.isna().sum())

No missing values in the columns we need (`subject_id`, `hadm_id`, `icd_code` in diagnoses; `subject_id`, `hadm_id`, `admittime` in admissions). Other admissions columns like `deathtime` and `discharge_location` have missing values, but we will not use those columns, so this is not a concern.

### 3c. Filtering Columns and Merging

We only need a few columns from each table. Filtering early reduces memory usage. We then merge on `subject_id` + `hadm_id` to attach the admission date to each diagnosis.

In [ ]:
print(f"Diagnoses shape before merge: {diagnoses_icd.shape}")

timed_diagnoses = pd.merge(
    diagnoses_icd[["subject_id", "hadm_id", "icd_code", "icd_version"]],
    admissions[["subject_id", "hadm_id", "admittime"]],
    how="left",
    on=["subject_id", "hadm_id"],
)

print(f"Diagnoses shape after merge: {timed_diagnoses.shape}")
display(timed_diagnoses.head())

The shape stayed the same after the merge, confirming that every diagnosis record matched exactly one admission. This is expected because diagnoses are recorded per admission.

After any merge, it is good practice to check for new missing values. A failed merge (for example, due to mismatched data types between `subject_id` columns) would show up as nulls in the joined columns.

In [ ]:
print(f"Shape before dropping nulls: {timed_diagnoses.shape}")

timed_diagnoses = timed_diagnoses.dropna(subset=["subject_id", "icd_code", "icd_version", "admittime"])

print(f"Shape after dropping nulls: {timed_diagnoses.shape}")

No rows were lost, confirming every diagnosis has a valid patient ID, code, and date.

### 3d. Standardizing the Schema

Now we transform the data into our standard four-column format. We truncate timestamps to just the date (we only need day-level precision for this analysis), rename columns, and add a `coding_system` label. We do this so that all five datasets share the same column names, making downstream code simpler.

In [ ]:
# Truncate timestamp to date only (YYYY-MM-DD)
timed_diagnoses["admittime"] = timed_diagnoses["admittime"].str[:10]

# Build the coding_system column: "ICD" + version number (e.g., "ICD9", "ICD10")
timed_diagnoses["coding_system"] = "ICD" + timed_diagnoses["icd_version"]

# Select and rename to standard schema
timed_diagnoses = timed_diagnoses.rename(columns={"admittime": "date", "icd_code": "code"})
timed_diagnoses = timed_diagnoses[["subject_id", "date", "code", "coding_system"]]

display(timed_diagnoses.head())

Notice the `coding_system` column distinguishes ICD9 from ICD10 codes. We do this because the same condition can have different codes under each version, and the downstream code rollup notebook needs to know which mapping table to use.

> **Note:** We keep dates as strings during processing to keep the logic straightforward. For time-series analysis (e.g., ECG data), you would preserve the full timestamp instead of truncating.

### 3e. Removing Duplicates

Duplicate records are common in EHR data and can arise as **exact duplicates** (identical rows repeated) or **semantic duplicates** (different codes representing the same clinical concept). We remove exact duplicates now and will deduplicate again after code rollup in the next notebook, since different raw codes can map to the same parent code.

In [ ]:
print(f"Shape before deduplication: {timed_diagnoses.shape}")

timed_diagnoses = timed_diagnoses.drop_duplicates()

print(f"Shape after deduplication: {timed_diagnoses.shape}")

No duplicates found in diagnoses. Other datasets (especially labs, where the same test may be recorded multiple times per day) may have duplicates that get removed in this step.

That completes the walkthrough. Every cleaning step followed the same pattern: check the data, transform it, verify the result. Next, we will scale this up to handle large datasets using batch processing.

## 4. Why Batch Processing?

The lab dataset alone has over **158 million rows**, which is too large to load into memory at once. Even smaller datasets benefit from batching because batch files can be processed in parallel on a computing cluster.

We batch at the **patient level** so that all records for a given patient stay together in the same batch. This is important because deduplication needs to see all of a patient's records at once. We split the ~223,000 unique patients into batches of roughly equal size.

In [ ]:
admissions = pd.read_csv(os.path.join(hosp_dir, "admissions.csv"), dtype=str)

# Extract unique patient IDs and sort them for reproducibility
patient_batches = admissions[["subject_id"]].drop_duplicates().sort_values("subject_id").reset_index(drop=True)

# Adjust this if you have more or less memory available
num_batches = 8

# Assign each patient to a batch using round-robin (modulo)
patient_batches["batch_num"] = (patient_batches.index % num_batches) + 1

print(f"Total unique patients: {len(patient_batches)}")
print(f"\nPatients per batch:")
print(patient_batches.groupby("batch_num")["subject_id"].count().to_string())

This tells us that ~28,000 patients are assigned to each of the 8 batches. The round-robin assignment ensures roughly equal batch sizes.

## 5. Building the Reusable Cleaning Function

We now wrap the walkthrough steps into a single reusable function. Compare each section of the function to the inline steps above -- it follows the same sequence:

1. Read the input file in chunks (to handle large files like labs)
2. Filter each chunk to the current batch's patients
3. Drop rows with missing values in key columns
4. Truncate timestamps to dates
5. Rename columns to the standard schema
6. Add the `coding_system` label
7. Remove duplicates
8. Save the cleaned batch to a CSV file

In [ ]:
def clean_data_by_batch(input_file, output_dir, patient_batches,
                        code_col, date_col, coding_system_label,
                        data_name, code_version_col=None,
                        chunk_size=15_000_000):
    """
    Clean an EHR dataset in patient-level batches.

    Parameters:
        input_file: path to the raw CSV file
        output_dir: directory to save cleaned batch files
        patient_batches: DataFrame with subject_id and batch_num columns
        code_col: name of the code column (e.g., "icd_code")
        date_col: name of the date/time column (e.g., "admittime")
        coding_system_label: label for the coding system (e.g., "ICD")
        data_name: name for output folder and files (e.g., "Diagnoses")
        code_version_col: optional column with version info (e.g., "icd_version")
        chunk_size: number of rows to read per chunk
    """
    source_name = os.path.splitext(os.path.basename(input_file))[0]
    data_output_dir = os.path.join(output_dir, data_name)
    os.makedirs(data_output_dir, exist_ok=True)

    # Build the list of columns to read from the CSV
    usecols = ["subject_id", date_col, code_col]
    if code_version_col:
        usecols.append(code_version_col)

    unique_batches = sorted(patient_batches["batch_num"].unique())

    for batch_num in tqdm(unique_batches, desc=f"Cleaning {data_name}", unit="batch"):
        # Collect rows belonging to this batch across all chunks
        batch_patients = patient_batches[patient_batches["batch_num"] == batch_num]
        collected = []

        for chunk in pd.read_csv(input_file, chunksize=chunk_size, usecols=usecols, dtype=str):
            # Filter to patients in this batch
            matched = chunk.merge(batch_patients[["subject_id"]], on="subject_id", how="inner")
            if not matched.empty:
                collected.append(matched)

        if not collected:
            print(f"Warning: No data found for batch {batch_num}")
            continue

        # Combine all chunks for this batch
        batch_df = pd.concat(collected, ignore_index=True)
        print(f"Batch {batch_num}: {batch_df.shape[0]} rows loaded")

        # Drop rows missing key columns
        batch_df = batch_df.dropna(subset=["subject_id", date_col, code_col])

        # Truncate timestamp to date (YYYY-MM-DD)
        batch_df[date_col] = batch_df[date_col].str[:10]

        # Rename to standard schema
        batch_df = batch_df.rename(columns={date_col: "date", code_col: "code"})

        # Add coding_system label
        if code_version_col:
            batch_df["coding_system"] = coding_system_label + batch_df[code_version_col]
        else:
            batch_df["coding_system"] = coding_system_label

        # Keep only standard columns and remove duplicates
        batch_df = batch_df[["subject_id", "date", "code", "coding_system"]]
        rows_before = batch_df.shape[0]
        batch_df = batch_df.drop_duplicates()
        rows_removed = rows_before - batch_df.shape[0]
        if rows_removed > 0:
            print(f"  Removed {rows_removed} duplicate rows")

        # Save to CSV
        output_file = os.path.join(data_output_dir, f"{data_name.lower()}_batch{batch_num}_{source_name}.csv")
        batch_df.to_csv(output_file, index=False)

    print(f"\nDone! All batches saved to: {data_output_dir}")

The function uses **named parameters** (`code_col`, `date_col`, `coding_system_label`) instead of a dictionary, so it is clear what each argument controls. The optional `code_version_col` parameter handles datasets like diagnoses and ICD procedures where the coding system depends on a version column (e.g., `"ICD" + "9"` = `"ICD9"`).

## 6. Cleaning Diagnoses

Diagnoses is the only dataset where the code and timestamp live in separate files. We merge them first and save to a temporary CSV, then pass that file to our cleaning function. We do this because the function reads from disk in chunks, which is essential for the large labs file, and using a consistent interface keeps the code simple.

In [ ]:
diagnoses_icd = pd.read_csv(os.path.join(hosp_dir, "diagnoses_icd.csv"), dtype=str)
admissions = pd.read_csv(os.path.join(hosp_dir, "admissions.csv"), dtype=str)

print(f"Diagnoses shape before merge: {diagnoses_icd.shape}")

timed_diagnoses = pd.merge(
    diagnoses_icd,
    admissions[["subject_id", "hadm_id", "admittime"]],
    how="left",
    on=["subject_id", "hadm_id"],
)

print(f"Diagnoses shape after merge: {timed_diagnoses.shape}")

# Save merged file so the batch function can read it in chunks
tmp_dir = os.path.join(base_dir, "scripts", "tmp")
os.makedirs(tmp_dir, exist_ok=True)
timed_diagnoses_file = os.path.join(tmp_dir, "timed_diagnoses_icd.csv")
timed_diagnoses.to_csv(timed_diagnoses_file, index=False)

In [ ]:
clean_data_by_batch(
    input_file=timed_diagnoses_file,
    output_dir=cleaned_data_dir,
    patient_batches=patient_batches,
    code_col="icd_code",
    date_col="admittime",
    coding_system_label="ICD",
    data_name="Diagnoses",
    code_version_col="icd_version",
)

## 7. Cleaning Procedures

Procedure data in MIMIC-IV comes from two separate sources:

1. **`hcpcsevents.csv`** -- procedures recorded using HCPCS/CPT billing codes
2. **`procedures_icd.csv`** -- procedures recorded using ICD-9 and ICD-10 procedure codes

We clean each one separately because they use different coding systems. Both are saved to the same `Procedures/` output folder so they can be combined during downstream analysis.

### HCPCS Procedures

HCPCS (Healthcare Common Procedure Coding System) codes are used for billing and include CPT codes for medical services. Unlike diagnoses, these already have a `chartdate` column, so no pre-merge is needed.

In [ ]:
hcpcs_file = os.path.join(hosp_dir, "hcpcsevents.csv")

print("HCPCS sample:")
display(pd.read_csv(hcpcs_file, nrows=5, dtype=str))

In [ ]:
clean_data_by_batch(
    input_file=hcpcs_file,
    output_dir=cleaned_data_dir,
    patient_batches=patient_batches,
    code_col="hcpcs_cd",
    date_col="chartdate",
    coding_system_label="HCPCS",
    data_name="Procedures",
)

### ICD Procedures

ICD procedure codes are recorded with a version column (`icd_version`), similar to diagnoses. The `coding_system` will be `ICDPROC9` or `ICDPROC10` depending on the version.

In [ ]:
procedures_icd_file = os.path.join(hosp_dir, "procedures_icd.csv")

print("ICD Procedures sample:")
display(pd.read_csv(procedures_icd_file, nrows=5, dtype=str))

In [ ]:
clean_data_by_batch(
    input_file=procedures_icd_file,
    output_dir=cleaned_data_dir,
    patient_batches=patient_batches,
    code_col="icd_code",
    date_col="chartdate",
    coding_system_label="ICDPROC",
    data_name="Procedures",
    code_version_col="icd_version",
)

## 8. Cleaning Medications

Prescriptions use **NDC** (National Drug Code) as their code identifier and `starttime` as the timestamp. NDC is a unique 10-digit code assigned to each medication product in the United States.

In [ ]:
prescriptions_file = os.path.join(hosp_dir, "prescriptions.csv")

print("Prescriptions sample:")
display(pd.read_csv(prescriptions_file, nrows=5, dtype=str))

In [ ]:
clean_data_by_batch(
    input_file=prescriptions_file,
    output_dir=cleaned_data_dir,
    patient_batches=patient_batches,
    code_col="ndc",
    date_col="starttime",
    coding_system_label="NDC",
    data_name="Medication",
)

## 9. Cleaning Labs

Laboratory data is the **largest** dataset in MIMIC-IV, with over 158 million rows. This is where batch processing truly matters -- loading the entire file at once would exceed most machines' memory. The cleaning function reads 15 million rows at a time and processes each patient batch across all chunks.

Labs use `itemid` as the code (a MIMIC-specific identifier for each lab test) and `charttime` as the timestamp.

> **Expect this step to take approximately 30 minutes.** If it runs too slowly, try reducing `chunk_size`. If you have access to a computing cluster, this is a good candidate for parallelization (see the Advanced Module for SLURM templates).

In [ ]:
labevents_file = os.path.join(hosp_dir, "labevents.csv")

print("Lab events sample:")
display(pd.read_csv(labevents_file, nrows=5, dtype=str))

In [ ]:
clean_data_by_batch(
    input_file=labevents_file,
    output_dir=cleaned_data_dir,
    patient_batches=patient_batches,
    code_col="itemid",
    date_col="charttime",
    coding_system_label="ITEMID",
    data_name="Labs",
)

## 10. Validating the Output

Before moving on, we verify that every batch file was created and the output schema is consistent. This catches problems early -- for example, a file that was not written because of an error, or a column that was accidentally dropped.

In [ ]:
expected_folders = ["Diagnoses", "Procedures", "Medication", "Labs"]
expected_schema = ["subject_id", "date", "code", "coding_system"]

for folder in expected_folders:
    folder_path = os.path.join(cleaned_data_dir, folder)
    files = sorted([f for f in os.listdir(folder_path) if f.endswith(".csv")])
    print(f"\n{folder}: {len(files)} batch files")
    for f in files:
        print(f"  {f}")

Now we spot-check one batch file from each data type to verify the schema is correct.

In [ ]:
for folder in expected_folders:
    folder_path = os.path.join(cleaned_data_dir, folder)
    first_file = sorted([f for f in os.listdir(folder_path) if f.endswith(".csv")])[0]
    sample = pd.read_csv(os.path.join(folder_path, first_file), dtype=str, nrows=3)

    print(f"\n--- {folder} ({first_file}) ---")
    print(f"Columns: {list(sample.columns)}")
    print(f"Schema matches: {list(sample.columns) == expected_schema}")
    display(sample)

All batch files exist and share the same four-column schema: `subject_id`, `date`, `code`, `coding_system`. The output directory structure looks like this:

```
EHR_TUTORIAL_WORKSPACE/
└── processed_data/
    └── step3_cleaned_rawdata/
        ├── Diagnoses/
        │   ├── diagnoses_batch1_timed_diagnoses_icd.csv
        │   ├── ... (8 batch files)
        │   └── diagnoses_batch8_timed_diagnoses_icd.csv
        ├── Procedures/
        │   ├── procedures_batch1_hcpcsevents.csv
        │   ├── procedures_batch1_procedures_icd.csv
        │   ├── ... (8 batches x 2 sources)
        │   └── procedures_batch8_procedures_icd.csv
        ├── Medication/
        │   ├── medication_batch1_prescriptions.csv
        │   ├── ... (8 batch files)
        │   └── medication_batch8_prescriptions.csv
        └── Labs/
            ├── labs_batch1_labevents.csv
            ├── ... (8 batch files)
            └── labs_batch8_labevents.csv
```

## 11. What We Accomplished

In this notebook, we:

- **Walked through five cleaning steps** on the diagnoses dataset: assessing missingness, filtering columns, standardizing the schema, checking for erroneous data, and removing duplicates.
- **Built a single reusable function** (`clean_data_by_batch`) that applies these steps to any EHR dataset while processing data in memory-efficient patient-level batches.
- **Cleaned all five MIMIC-IV datasets** (diagnoses, HCPCS procedures, ICD procedures, medications, and labs) into a standardized four-column CSV format.
- **Validated the output** by verifying that all batch files exist and share the correct schema.

### A Note on Erroneous Data

In real-world (non-MIMIC) EHR datasets, you should also check for **erroneous data** before saving your cleaned files:

- **Invalid dates:** Filter out records with dates before a reasonable start year (e.g., 1980) or after the current year. These often indicate data entry errors.
- **Impossible lab values:** Some lab results may have clearly impossible values (e.g., a negative hemoglobin level). These can be safely removed.
- **Unit mismatches:** The same lab test may be recorded in different units (e.g., hemoglobin in g/dL vs. mmol/L). Normalizing units requires domain knowledge but is essential for consistent analysis.

We skip these checks for MIMIC-IV because the dates have been shifted for de-identification and we do not use lab values in this tutorial. When working with your own data, add these validation steps after the schema standardization step.

**Next:** In the [Code Rollup notebook](github_repo/TUTORIAL/EHR_Preprocessing_2_Code_Rollup.ipynb), we will map these granular codes to standardized parent codes, reducing thousands of individual codes into clinically meaningful groups.

> **Full executable notebook:** [github_repo/TUTORIAL/EHR_Preprocessing_1_Data_Cleaning.ipynb](github_repo/TUTORIAL/EHR_Preprocessing_1_Data_Cleaning.ipynb)